# 02. JSON

**Tier:** Intermediate. **Why you'd use this:** An API gives you a JSON object as text. You need a Python dict, and you need to know that a tuple you dump will come back as a list.

**After this lesson you can:** `json.dumps` / `json.loads`; say how JSON types map to Python; keep keys as strings.
About 25 minutes. **Uses:** [Paths](01-paths-lesson.ipynb), [Dicts](../04-collections/03-dicts-lesson.ipynb)


## Quick review

1. Why keep `encoding="utf-8"` on text files?
2. What does `with` guarantee for an open file?

<details><summary>Answers</summary>

Stable meaning across machines. Close and flush even after errors.
</details>


## The idea

JSON is a text format. `json.dumps` turns Python data into that text. `json.loads` parses text back. Types do not match one-to-one: JSON objects become dicts with string keys, JSON arrays become lists, and Python tuples dump as arrays so they load as lists.

<details><summary>Explain more</summary>

`json.dumps({"n": 1})` is a string. Keys in JSON objects are always strings.
</details>



## See it work

Round-trip a dict and watch a tuple become a list.


In [1]:
import json

payload = {"name": "Ada", "scores": (1, 2)}
# 1. dumps makes text; loads parses it back.
text = json.dumps(payload)
print(text)
print(json.loads(text))


{"name": "Ada", "scores": [1, 2]}
{'name': 'Ada', 'scores': [1, 2]}


## Check yourself

Answer these before you run the cell.

1. What type is `json.loads('[1, 2]')`?
2. What happens to tuple values after dump and load?


In [2]:
import json

print(type(json.loads("[1, 2]")).__name__)
print(json.loads(json.dumps({"pair": (1, 2)})))


list
{'pair': [1, 2]}


<details><summary>Answers</summary>

`list`. The tuple becomes a list.
</details>


## Why it works this way

JSON is language-neutral text with a small type set: objects, arrays, strings, numbers, booleans, null. Python's `json` module maps those onto dict, list, str, int/float, bool, and `None`. Anything outside that set must be converted or it raises. Tuples are not a JSON type, so they serialize as arrays and come back as lists. Dict keys that are not strings are coerced to strings on dump, which can surprise you if you used int keys. Deep dive: keep the mapping table next to any API boundary.

`dumps` and `loads` work on strings in memory. `dump` and `load` work on file objects. Prefer explicit UTF-8 when the JSON lives on disk, because the file is still bytes underneath.

Cost is a full parse or serialize of the document. Streaming JSON needs other tools. The design reason for the type narrowing is interoperability: every language can agree on the small JSON set.

Evidence: a dumped tuple field reloads as a list, and object keys in text are quoted strings. When you need a tuple after load, convert deliberately.  Pretty-printing with `indent=` is for humans. Compact dumps are for wires. Either way the type mapping stays the same. Validate loaded data if you do not trust the source.

Numbers in JSON are not a full decimal type. Large integers may survive in Python, but other languages may treat them as floats. Booleans are lowercase in JSON text. `null` becomes `None`. When schema matters, validate after load. When performance matters, avoid dumping the same large object repeatedly in a tight loop.


In [3]:
import json

text = json.dumps({"name": "Ada", "pair": (1, 2)})
print(text)
data = json.loads(text)
print(data)
print(type(data["pair"]).__name__)
sample = json.loads('{"1": "a"}')
key = next(iter(sample))
print(key, type(key).__name__)


{"name": "Ada", "pair": [1, 2]}
{'name': 'Ada', 'pair': [1, 2]}
list
1 str


## When to use it

Use JSON for interoperable structured text. Convert tuples and non-string keys at the boundary. Don't expect Python-only types to survive a round trip. Don't hand-edit JSON with string replace when `json` can do it. Validate shapes at the edge when the sender is not your own process.


## Common mistake

Assuming a dumped tuple comes back as a tuple.


In [4]:
import json

data = json.loads(json.dumps({"pair": (1, 2)}))
print(data["pair"])
print(type(data["pair"]).__name__)


[1, 2]
list


Convert to `tuple` yourself if your code needs one after load.


In [5]:
import json

data = json.loads(json.dumps({"pair": (1, 2)}))
pair = tuple(data["pair"])
print(pair)
print(type(pair).__name__)


(1, 2)
tuple


## Try it

Dump a dict with an int key `1` and load it back. What type is the key after load?


## Recap

- `dumps`/`loads` move between Python data and JSON text.
- Tuples become lists. Object keys become strings.
- Convert at the boundary when you need richer Python types.

**Next:** [practice](02-json-practice.ipynb)